# A single black hole with ExaGRyPE: a first (too) trivial setup

<img src="https://upload.wikimedia.org/wikipedia/commons/5/5e/BH_LMC.png" width="60%"/>

This tutorial simulates a single, non-spinning black hole with ExaGRyPE. It
is the second step after @ref tutorials_exahype2_exagrype_521_gauge-wave.
The gauge wave is flat spacetime in disguise, while we now evolve a spacetime
with a genuine singularity. At the same time, the tutorial documents how we
built the first version of the benchmark @ref benchmarks_exagrype_single_black_hole
with the new Python API exagrype.api.

We run through the following steps:

- how to obtain black hole initial data from TwoPunctures, and why we wrap it
  into a C++ routine of the ExaGRyPE library rather than generating it from
  Python;
- how to impose boundary conditions on a domain which is not periodic;
- how to pick a regular Finite Volume mesh which avoids the singularity at the
  puncture;
- how to check the initial data against a closed formula, and how to observe
  the evolution of the lapse;
- how to use several cores via a domain decomposition, and whether a finer
  mesh improves the solution;
- how to push the solution back onto the CCZ4 constraints after each time
  step, and what Sommerfeld boundary conditions are.

Before you start, configure and build Peano as described in
@ref tutorials_exahype2_applications_exagrype_configure. Different to the
gauge wave, you need the GNU Scientific Library (GSL), i.e. you have to
configure with `--with-gsl` on top of `--enable-exagrype`. TwoPunctures,
which computes our initial data, relies on it.

## The single black hole

We simulate the Schwarzschild solution, i.e. a black hole of mass $M$ which
neither moves nor spins. In isotropic coordinates, its spatial metric is
conformally flat:

\begin{equation*}
\gamma_{ij} = \psi^4 \delta_{ij},
\qquad
\psi(r) = 1 + \frac{M}{2r}.
\end{equation*}

The conformal factor $\psi$ diverges at $r=0$. This point is called the
puncture. ExaGRyPE uses the puncture method: Rather than cutting the black
hole out of the domain (excision), it evolves the singular data and relies on
the gauge, i.e. on lapse and shift, to keep the numerical solution under
control around the puncture. Inside the black hole, the details of the
solution do not matter, as no information escapes through the horizon at
$r = M/2$. The price to pay is a high resolution requirement around the
puncture.

We place the black hole with $M=1$ at the origin of the domain
$[-9,9]^3$. The data is not
exactly what an analytical formula gives us: Black hole simulations
typically obtain their initial data from TwoPunctures, a solver for binary
black hole initial data from the Einstein Toolkit. For a single black hole,
we set the mass of the second black hole to zero. We will check below that
TwoPunctures then indeed reproduces the formula above.

What do we expect? The black hole is stationary, but our coordinates are not
adapted to it: The lapse and the shift evolve until they settle into a
stationary gauge. The system should hence become (quasi-)static after some
time, with the black hole staying at the origin. A real production run
therefore runs for a long time and checks that the black hole stays where it
is and that the solution settles down.

## Configuring the environment

As in the gauge wave tutorial, we add the Python API to the search path and
memorise the Peano root directory.

In [ ]:
import sys, os

peano_root = os.path.abspath("../../../")
sys.path.insert(0, os.path.join(peano_root, "src"))

import peano4.api
import exahype2.api
import exagrype.api

## Step 1: Initial data from TwoPunctures

The initial condition of an exahype2.api.Solver is an
exahype2.api.InitPointWisely term: a C++ snippet which receives the position
x of one degree of freedom and writes the initial values of all unknowns into
Q. For the gauge wave, this snippet calls an analytical formula from the
ExaGRyPE library. TwoPunctures is different, as it is a solver of its own: It
first has to solve an elliptic equation on a spectral grid. Afterwards, we
can interpolate its solution at any point x.

Rather than generating the setup of TwoPunctures from Python, we moved all
of this logic into the ExaGRyPE library itself:
exagrype::initialvalues::singleBlackHole() (in
src/exagrype/initialvalues/ApplyTwoPunctures.h). The routine

- creates and solves a TwoPunctures object for one black hole of a given
  mass upon its first call. TwoPunctures always models two punctures at
  $\pm b$ along the x-axis. We shift them such that the first one sits at the
  origin, and give the second one zero mass;
- keeps this object in a function-local static variable. C++ guarantees that
  such a variable is initialised exactly once, even if several threads call
  the routine at the same time. This matters, as Peano computes the initial
  condition of many patches concurrently;
- interpolates the TwoPunctures solution at x and converts it into the 59
  unknowns of CCZ4, including the derivatives which the first-order
  formulation treats as unknowns of their own.

The Python counterpart exagrype.api.initialvalues.single_black_hole() then
is a one-liner: It returns an InitPointWisely term which calls the C++
routine. Let's look at the C++ snippet it produces:

In [ ]:
initial_condition = exagrype.api.initialvalues.single_black_hole(mass=1.0)
print(initial_condition.implementation)

TwoPunctures only becomes part of the ExaGRyPE library if Peano is configured
with GSL. Otherwise, the snippet stops the compilation with a meaningful
error message rather than an obscure linker error.

This is a pattern which we will see over and over again: Whenever the
generated code would contain more than a call or two, we write the logic as a
plain C++ function in the library, and the Python API only generates the call.
The C++ code then can be read, debugged and unit tested like any other C++
code.

## The solver

We create the CCZ4 solver as for the gauge wave, but hand in the black hole
initial condition. We also change two parameters: We switch on the
evolution of the shift (CCZ4sk), i.e. the Gamma driver, and
the advection terms within this driver (CCZ4bs). The gauge wave keeps the
shift at zero and hence does not need either of them. All other parameters
keep their defaults. In particular,
CCZ4LapseType=0 selects harmonic slicing.

This is not yet the setup of ExaGRyPE's production runs. They use 1+log
slicing (CCZ4LapseType=1), the standard choice for black holes, as it makes
the lapse collapse inside the horizon. We stick to harmonic slicing in this
tutorial and the next one. Furthermore, the Finite Volumes which we use
below evolve CCZ4 without the advection terms $\beta^k \partial_k Q$ of most
unknowns: exagrype::ccz4::ncp() omits them and expects the numerical scheme
to add them, which the Finite Volumes do not. Of the advection terms which
CCZ4bs switches on, only the one of the shift itself remains, as it is part
of the algebraic source.
@ref tutorials_exahype2_exagrype_523_higher-order-fd adds the missing terms
to the Finite Differences.

In [ ]:
project = exahype2.api.Project(
    namespace    = ["tutorials", "exagrype", "singleblackhole"],
    project_name = "SingleBlackHole",
    directory    = "522_single-black-hole",
    executable   = "single-black-hole",
)

black_hole_mass = 1.0

solver = exagrype.api.CCZ4Solver(
    "CCZ4",
    parameters        = {"CCZ4bs": 1.0, "CCZ4sk": 1.0},
    initial_condition = exagrype.api.initialvalues.single_black_hole(mass=black_hole_mass),
)
solver.parameters

## Step 2: Boundary conditions

The gauge wave lives on a periodic domain. The black hole does not: Our
domain $[-9,9]^3$ cuts out a finite piece of an infinite spacetime, and
we have to tell the code what happens at its boundary. As soon as at least
one axis is not periodic, exahype2.api expects the solver to have exactly
one boundary condition term: either an
exahype2.api.DirichletBoundaryConditions or an
exahype2.api.SommerfeldBoundaryConditions term. The Dirichlet term receives
the position x of a ghost volume outside of the domain and a time stamp t,
and it has to write the solution there into Q.

Black hole codes typically use Sommerfeld conditions, which we discuss at
the end of this tutorial. They assume that all fields
approach their values at infinity like outgoing waves, i.e. they let waves
leave the domain with little reflection. Sommerfeld conditions need the
solution inside the domain and the normal of the boundary face. The
Dirichlet term of the new API receives neither of them. For a first version,
we hence take the simplest option that is consistent with our setup: We hold
the boundary at the initial data. The spacetime far away from the black hole
hardly changes, so this is a reasonable approximation. However, everything
that travels outward will be reflected at the boundary.

In [ ]:
solver.add_pde_term(
    exahype2.api.DirichletBoundaryConditions(
        implementation="::exagrype::initialvalues::singleBlackHole(Q, x, t, {});".format(black_hole_mass)
    )
)

## Numerical scheme and mesh

We use the Finite Volume scheme with a Rusanov flux and explicit Euler in
time, and a regular mesh. We pick patches of $6 \times 6 \times 6$ volumes
and $9 \times 9 \times 9$ mesh cells. Each cell then
has an edge length of 2, and each volume an edge length of $h=1/3$.

The patch size matters more than it might seem. Our domain is symmetric
around the puncture, and Peano splits every cell into $3 \times 3 \times 3$
children. The number of cells per axis is hence always odd, and the origin
always lies in the centre of a cell. With an odd patch size, it would also be
the centre of a volume, i.e. we would evaluate the initial data exactly at
the singularity. With an even patch size, the origin sits on a vertex shared
by eight volumes, and the closest volume centres are at
$(\pm h/2, \pm h/2, \pm h/2)$.

As explained in the gauge wave tutorial, Peano refines every cell as long as
its edge length is greater than or equal to the maximum volume size times the
patch size. We therefore hand in a maximum volume size which is slightly
bigger than the volume size we want to get.

We use exahype2.api.fv.RusanovWithExplicitEuler, i.e. ExaHyPE's plain
Finite Volume scheme. At the end of the tutorial, we try a faster variant of
this scheme and the enforcement of the CCZ4 constraints.

In [ ]:
patch_size     = 6
cells_per_axis = 9
domain_size    = 18.0

max_volume_size = 1.01 * domain_size / cells_per_axis / patch_size

solver.numerical_scheme  = exahype2.api.fv.RusanovWithExplicitEuler(solver, patch_size)
solver.grid_construction = exahype2.api.fv.SetupSimulationOverRegularGrid(solver, volume_size=max_volume_size)

## Plotting

One snapshot of the whole domain holds $54^3 \cdot 59 \approx 9.3$ million
values. As the black hole is spherically symmetric, we only plot the row of
patches along the x-axis. The plot filter selects all patches which overlap
a thin box close to the line $y=z=1/6$. We plot every 0.5 time units. See
@ref tutorials_exahype2_applications_exagrype_visualize for remarks on how
to look at the output in ParaView.

In [ ]:
solver.plotter = exahype2.api.fv.PlotAsVTKFile(
    solver,
    first_plot_time_stamp = 0.0,
    time_in_between_plots = 0.5,
    plotter_precision     = 10,
)
solver.plotter.add_plot_filter(offset=[-9.0, 0.1, 0.1], size=[18.0, 0.1, 0.1])

## Global simulation parameters

We simulate until $t=10$, i.e. for $10 M$. None of the axes is periodic.

In [ ]:
end_time = 10.0

project.add_solver(solver)

project.set_global_simulation_parameters(
    dimensions   = 3,
    offset       = [-9.0, -9.0, -9.0],
    size         = [domain_size, domain_size, domain_size],
    min_end_time = end_time,
    max_end_time = end_time,
    periodic_BC  = [False, False, False],
)

## Lowering into C++

The lowering is exactly the same as for the gauge wave:

In [ ]:
project.set_Peano4_installation(peano_root, peano4.api.output.CompileMode.RELEASE)

peano4_project = project.generate_Peano4_project(verbosity=peano4.api.LoggerVerbosity.TEXT)
peano4_project.generate()

Both the initial condition and the boundary conditions end up as calls of
the same library routine in AbstractCCZ4.cpp:

In [ ]:
!grep -n "singleBlackHole" 522_single-black-hole/AbstractCCZ4.cpp

We reuse the log filter of the gauge wave tutorial and only switch on the
info messages:

In [ ]:
%%writefile 522_single-black-hole/exahype2.log-filter
debug     tarch       -1      black     always-on
debug     peano4      -1      black     always-on
debug     exahype2    -1      black     always-on
debug     exagrype    -1      black     always-on
debug     toolbox     -1      black     always-on
debug     tutorials   -1      black     always-on

trace     tarch       -1      black     always-on
trace     peano4      -1      black     always-on
trace     exahype2    -1      black     always-on
trace     exagrype    -1      black     always-on
trace     toolbox     -1      black     always-on
trace     tutorials   -1      black     always-on

info      tarch       -1      white     always-on
info      peano4      -1      white     always-on
info      exahype2    -1      white     always-on
info      exagrype    -1      white     always-on
info      toolbox     -1      white     always-on
info      tutorials   -1      white     always-on

## Build and run

The executable first solves the TwoPunctures problem. You see its output
right at the start of the initial condition phase. Afterwards, the code runs
for around 200 time steps. On an eight-core laptop, the run takes around
seven minutes.

In [ ]:
!cd 522_single-black-hole && make -j4

In [ ]:
!cd 522_single-black-hole && ./single-black-hole

## Checking the initial data

We extract the solution along the x-axis with Peano's PointProbe, as in the
gauge wave tutorial. We place one probe into the centre of each of the 54
volumes along the line $y=z=h/2=1/6$, and read the lapse and the conformal
factor phi:

In [ ]:
import contextlib
import csv
import io
import numpy as np
import matplotlib.pyplot as plt

import tarch.plotter.api.output


def probe(point, output_file, directory="522_single-black-hole"):
    point_probe = tarch.plotter.api.output.PointProbe(
        file_name   = os.path.join(directory, "solution-CCZ4.peano-patch-file"),
        point       = point,
        output_file = os.path.join(directory, output_file),
        start       = 0,
        end         = sys.maxsize,
    )
    with contextlib.redirect_stdout(io.StringIO()):
        point_probe.display()
    with open(os.path.join(directory, output_file)) as csv_file:
        rows = list(csv.DictReader(csv_file))
    t     = np.array([float(row["time"])  for row in rows])
    lapse = np.array([float(row["lapse"]) for row in rows])
    phi   = np.array([float(row["phi"])   for row in rows])
    return t, lapse, phi


h = domain_size / cells_per_axis / patch_size
x = -9.0 + (np.arange(54) + 0.5) * h
lapse, phi = [], []
for i, x_i in enumerate(x):
    t, lapse_over_time, phi_over_time = probe([x_i, h / 2, h / 2], "profile-{}.csv".format(i))
    lapse.append(lapse_over_time)
    phi.append(phi_over_time)
lapse = np.array(lapse).T
phi   = np.array(phi).T

The unknown phi is the conformal factor of the CCZ4 formulation. For our
metric, it equals $\psi^{-2}$. The initial lapse is a choice: TwoPunctures'
default, the averaged lapse, yields $\alpha = 1/\psi$ for a single black
hole. Note that the probe line does not pass through the puncture, i.e. the
distance of a probe from the puncture is $r = \sqrt{x^2 + 2 (h/2)^2}$:

In [ ]:
def psi(r):
    return 1.0 + black_hole_mass / 2.0 / r

r      = np.sqrt(x**2 + 2.0 * (h / 2)**2)
x_fine = np.linspace(-9.0, 9.0, 2000)
r_fine = np.sqrt(x_fine**2 + 2.0 * (h / 2)**2)

plt.plot(x_fine, 1.0 / psi(r_fine), "-",  label="1/psi")
plt.plot(x_fine, psi(r_fine)**(-2), "-",  label="psi^-2")
plt.plot(x, lapse[0], "o", markersize=3, label="lapse")
plt.plot(x, phi[0],   "s", markersize=3, label="phi")
plt.xlabel("x")
plt.legend()
plt.show()

print("max deviation of lapse from 1/psi: ", np.max(np.abs(lapse[0] - 1.0 / psi(r))))
print("max deviation of phi from psi^-2:  ", np.max(np.abs(phi[0] - psi(r)**(-2))))

![Initial lapse and conformal factor along the x-axis](illustrations/522_single-black-hole-initial-data.png)

The values of all 54 volumes agree with the closed formulas up to
around 5e-12. TwoPunctures hence indeed gives us a Schwarzschild black
hole at the origin. Both quantities drop towards zero at the puncture, where
$\psi$ diverges. Even the volumes next to the puncture are fine, as their
centres keep a distance of $\sqrt{3} h/2 \approx 0.29$ from it. The dashed
lines are the boundaries of the patches.

## The evolution of the lapse

We now look at the lapse over time. First, along the x-axis for a few
snapshots:

In [ ]:
for snapshot in [0, len(t) // 4, len(t) // 2, 3 * len(t) // 4, len(t) - 1]:
    plt.plot(x, lapse[snapshot], "o-", markersize=2, linewidth=1, label="t={:.2f}".format(t[snapshot]))
plt.xlabel("x")
plt.ylabel("lapse")
plt.legend()
plt.show()

![Lapse along the x-axis over time](illustrations/522_single-black-hole-lapse-profile.png)

and then over time at a few distances from the puncture:

In [ ]:
for x_probe in [h / 2, 1.0 + h / 2, 3.0 + h / 2, 8.0 + h / 2]:
    i = np.argmin(np.abs(x - x_probe))
    plt.plot(t, lapse[:, i], "o-", markersize=2, linewidth=1, label="r={:.2f}".format(r[i]))
plt.xlabel("t")
plt.ylabel("lapse")
plt.legend()
plt.show()

![Lapse over time at several distances from the puncture](illustrations/522_single-black-hole-lapse-over-time.png)

The run is stable: there are no oscillations and no NaNs, and far away from
the black hole, the lapse hardly changes. Close to the puncture, however, the
solution does not behave like a resolved black hole. With the 1+log slicing
of production runs, the lapse collapses towards zero inside the horizon, such
that the evolution freezes there. Harmonic slicing, which we use here, avoids
the singularity less strongly. Instead of a collapse, however, we observe that
the dip of the lapse fills up. At
the volume next to the puncture, the lapse grows from 0.37 to 0.88 by $t=10$,
and the profile along the x-axis flattens out. The conformal factor phi
behaves the same way: Next to the puncture, it grows from 0.13 to 0.93,
i.e. almost to its value at the boundary of the domain.

One likely reason is the resolution in combination with the scheme.
The horizon has a radius of 0.5, i.e. it spans about three volumes. The
Rusanov flux is first order, and, as we saw for the gauge wave, it adds a lot
of numerical diffusion on coarse meshes. Here, it diffuses the steep
profiles around the puncture away, and the black hole dissolves numerically.
The boundary conditions, which reflect everything that travels outward, might
add to this. Two further differences to a production run might contribute, too,
and this tutorial does not separate them from the resolution: the harmonic
slicing, and the advection terms which the Finite Volumes omit. We check below
that the missing constraint enforcement does not contribute.

This is the baseline which we have to improve upon. A resolved black hole
requires a much finer mesh around the puncture, which codes typically obtain
by refining the mesh adaptively towards the black hole. A regular mesh of
such a resolution is prohibitively expensive, which is why adaptive mesh
refinement is a key extension of this setup.

## Speeding up

So far, our run is effectively serial: Without load balancing, Peano keeps
the whole mesh in one tree, and only one thread traverses it. Before we
refine the mesh, we hence make the code use several cores.

### Domain decomposition

@ref tutorials_exahype2_30_domain-decomposition explains how Peano splits the
mesh into several trees, and how a load balancing strategy decides on these
splits. We use the same off-the-shelf heuristic, which splits the mesh once
its construction has stagnated, such that each thread gets one tree:

At the same time, we refine the mesh. We keep the $9 \times 9 \times 9$ mesh
cells, but use patches of $10 \times 10 \times 10$ volumes, i.e. we pass

     patch_size = 10

to the numerical scheme and a maximum volume size of
$1.01 \cdot 18/90$ to the grid construction. Each volume then has an edge
length of $h=0.2$ rather than $1/3$, i.e. the mesh is 1.67 times finer along
each axis and holds $90^3$ rather than $54^3$ volumes. The patch size remains
even, so the puncture still sits on a vertex, and the closest volume
centres keep a distance of $\sqrt{3} h/2 \approx 0.17$ from it. A finer mesh
requires smaller time steps, too, i.e. we pay twice: for more volumes and for
more time steps.

We constrain OpenMP to four threads, i.e. the load balancing creates four
trees:

     !cd 522_single-black-hole && OMP_NUM_THREADS=4 ./single-black-hole

The log shows the split: SpreadOutOnceGridStagnates cuts the 729 mesh cells
into four trees with around 182 cells each. The run now needs 338 instead of
203 time steps, and each time step takes 3.4 instead of 2.1 seconds, although
it updates 4.6 times as many volumes. The whole run takes around 20 minutes
on our laptop. Per volume update, the wall clock time drops from
$13.1\,\mu s$ to $4.7\,\mu s$, i.e. by a factor of 2.8. This factor is not a
pure parallel speedup: Besides the four trees, the bigger patches give each
patch update more volumes to work on.

![Lapse next to the puncture and along the x-axis for both meshes](illustrations/522_single-black-hole-speeding-up.png)

The finer mesh helps a little, but it does not change the picture. The volume
next to the puncture now sits at $r=0.17$ rather than $0.29$, so its initial
lapse is lower, i.e. 0.26 rather than 0.37. Yet, it fills up again, to 0.86
by $t=10$. Along the x-axis, the dip at $t=10$ is slightly deeper than on the
coarse mesh, 0.86 rather than 0.88, but the profile still flattens out. The
horizon now spans 2.5 volumes per radius, which is still far from resolving
it.

### Further tuning

This might be the right point to tune the code further. You could

1. try out the optimistic time stepping scheme
   exahype2.api.fv.RusanovWithOptimisticExplicitEuler, which computes parts of
   the next time step ahead of time;
2. switch to the xDSL kernels and employ GPUs, see
   @ref page_third-party_xDSL, @ref documentation_exahype2_dsl and
   @ref tutorials_exahype2_hpc_gpus; or
3. also enable MPI. The script remains unchanged: Configure Peano with MPI,
   and launch the executable with mpirun. The load balancing then distributes
   the trees among the ranks, too, see @ref page_exahype_domain-decomposition
   and @ref tutorials_exahype2_30_domain-decomposition.

## Improving the physics

We stick to the mesh of the previous section, i.e. to $90^3$ volumes which
four threads process, and now alter the physics rather than the performance.

### Enforcing the CCZ4 constraints

The CCZ4 formulation carries algebraic constraints: The conformal metric has
to have unit determinant, and the conformal extrinsic curvature has to be
trace-free. The exact solution obeys them, but the numerical solution drifts
away. ExaGRyPE can push the solution back onto the constraint surface after
every time step via exagrype::ccz4::enforceCCZ4constraints().

exagrype.api.CCZ4Solver does not do this by default. You switch the
enforcement on via enforce_ccz4_constraint(), which accepts one of three
values of exagrype.api.CCZ4Solver.EnforceCCZ4Constraint:

- No: never enforce the constraints. This is the default.
- OnlyFinalSolution: enforce them after each time step.
- IntermediateSteps: enforce them after each intermediate solution, too,
  i.e. after each stage of a Runge-Kutta scheme. For our Finite Volumes,
  this is the same as OnlyFinalSolution.

We add

The solver then inserts an additional step after each solution update of
the numerical scheme. It uses the mechanism which
@ref tutorials_exahype2_20_additional-solver-steps introduces. The task graph of the
time step gets an additional task "CCZ4_Enforce CCZ4 constraints after
Update cell", which applies the enforcement to each volume of the patch
before the solution is projected onto the faces. If you run the lowering
with TEXT logging, the output lists this task. The solver knows the
numerical schemes of ExaHyPE and where they write their solution. For any
other scheme, it refuses to enforce the constraints and raises an exception.

![Lapse and phi next to the puncture with and without constraint enforcement](illustrations/522_single-black-hole-constraints.png)

The enforcement hardly changes the outcome: Next to the puncture, the lapse
grows from 0.26 to 0.87 rather than 0.86, and phi from 0.07 to 0.92 rather
than 0.93. Along the whole x-axis, the lapse differs by at most 9e-3 from the
run without enforcement. The additional step does not cost measurable time
either: the run takes around 1100 seconds, which is within the variation of
our measurements. The dissolution of the black hole hence does not stem from a
drift away from the constraints.

### Sommerfeld boundary conditions

So far, we hold the boundary at the initial data, i.e. everything that
travels outward is reflected back into the domain. Sommerfeld conditions,
also called radiative or outgoing boundary conditions, avoid most of these
reflections. They assume that, close to the boundary, each field $u$ behaves
like an outgoing spherical wave which decays towards its value $u_\infty$ at
infinity:

\begin{equation*}
u(r,t) = u_\infty + \frac{f(r - v t)}{r},
\end{equation*}

with $v$ the speed at which the field propagates and $r$ the distance from
the centre of the system, i.e. from the point the waves emanate from. Differentiating this ansatz yields a condition
which the boundary has to satisfy,

\begin{equation*}
\partial_t u + v \frac{x_i}{r} \partial_i u + v \frac{u - u_\infty}{r} = 0,
\end{equation*}

i.e. the solution in the ghost volumes follows from the solution inside the
domain. This is the reason why our Dirichlet term cannot realise them: It
sees neither the interior solution nor the normal of the boundary face.

exahype2.api therefore offers a separate PDE term,
exahype2.api.SommerfeldBoundaryConditions, which you add instead of the
Dirichlet term. You specify the far-field solution $u_\infty$ and the
origin, while the wave speed $v$ is the maximum eigenvalue of your solver. For CCZ4, this is
an upper bound on the speeds of all unknowns, which evaluates to 2, i.e. to
twice the speed of light, in the far field.
There is no separate boundary task in this case: The ghost volumes are
advanced whenever the face data are rolled over after the projection of a
new solution
(@ref exahype2::blockstructured::applySommerfeldBoundaryConditions()).

To see what we gain, we let an acoustic pressure pulse of the Euler
equations leave the domain $[-1,1]^2$, discretised with $81$ grid cells per
axis, and compare the largest pressure deviation $\max |p-1|$ from the
far-field state that remains at $t=2$, once the pulse should have left the
domain. A uniform state equal to the far field has to stay exactly where it
is.

| Scheme | Uniform state | Pulse, Sommerfeld | Pulse, Dirichlet |
|---|---|---|---|
| Fourth-order Finite Differences (RK4) | exact up to round-off | $4.9 \cdot 10^{-4}$ | $5.3 \cdot 10^{-2}$ |
| Finite Volumes (Rusanov) | exact | $9.5 \cdot 10^{-4}$ | $7.6 \cdot 10^{-4}$ |
| Finite Volumes with optimistic time stepping | exact | $9.6 \cdot 10^{-4}$ | - |

For the fourth-order Finite Differences, the Sommerfeld conditions reduce
the reflections by two orders of magnitude. The Finite Volume runs do not
tell the two conditions apart: With a Rusanov flux, a ghost volume which is
held at the far-field state already absorbs small waves, and the
first-order scheme has damped the pulse considerably before it reaches the
boundary.

To clean up, delete the generated directory. If you rerun the notebook, it
is re-created.

In [ ]:
!rm -rf 522_single-black-hole